<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/HousePredictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn import metrics
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer,SimpleImputer

data = {
    'Rooms': [3, 4, np.nan, 2, 4, np.nan, 3, 5],
    'Age': [15, np.nan, 25, 5, np.nan, 30, 12, 8],
    'Income_Level': [80, 110, 65, np.nan, 95, 50, 85, np.nan], # Numerical
    'Neighborhood': ['Suburb', 'City', 'Rural', 'Suburb', 'City', 'Rural', 'Suburb', 'City'], # Categorical
    'Price': [250, 320, 190, 150, 310, 175, 240, 400] # Target variable
}

df = pd.DataFrame(data)
X = df.drop('Price', axis=1)
Y = df['Price']

# Identify column types automatically
numerical_cols = ['Rooms', 'Age', 'Income_Level']
categorical_cols = ['Neighborhood']

# Split into train and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)



numeric_transformer = Pipeline(steps=[
    ('mice_imputer', IterativeImputer(max_iter=10, random_state=0)), # MICE implementation
    ('scaler', StandardScaler())
])


categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough' # Keep any other unmentioned columns intact
)



final_pipeline = Pipeline(steps=[
    ('preprocessing_stage', preprocessor),
    ('xgboost_model', XGBRegressor(n_estimators=100, random_state=42))
])


print("--- Training the Pipeline (MICE Imputation + Preprocessing + Modeling) ---")

final_pipeline.fit(X_train, Y_train)
test_predictions = final_pipeline.predict(X_test)
r2 = metrics.r2_score(Y_test, test_predictions)
mae = metrics.mean_absolute_error(Y_test, test_predictions)

print(f"\nTest Evaluation Metrics:")
print(f"R-squared Score   : {r2}")
print(f"Mean Absolute Error: {mae}")

--- Training the Pipeline (MICE Imputation + Preprocessing + Modeling) ---

Test Evaluation Metrics:
R-squared Score   : -3.825129985809326
Mean Absolute Error: 117.49919128417969


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
